# Projeto Final Aplicado a Ciência de Dados
## Deteção de Outliers em Contratos Públicos e Previsão do Montante de Adjudição

## Grupo 08
## Membros:
###          - Afonso Alexandre nº 123425;
###          - André  Freitas nº 123409;
###          - Gonçalo Vilhena nº 123384;
###          - Rodolfo Nardy nº 123419




In [39]:
import pandas as pd
import numpy as np
import re
import unicodedata

In [40]:
file_path = "contratos_total.csv"
df = pd.read_csv(file_path, low_memory=False)


In [41]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1843246 entries, 0 to 1843245
Data columns (total 35 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   idcontrato                int64  
 1   nAnuncio                  object 
 2   TipoAnuncio               object 
 3   idINCM                    float64
 4   tipoContrato              object 
 5   idprocedimento            int64  
 6   tipoprocedimento          object 
 7   objectoContrato           object 
 8   descContrato              object 
 9   adjudicante               object 
 10  adjudicatarios            object 
 11  dataPublicacao            object 
 12  dataCelebracaoContrato    object 
 13  precoContratual           float64
 14  CPV                       object 
 15  prazoExecucao             int64  
 16  LocalExecucao             object 
 17  fundamentacao             object 
 18  ProcedimentoCentralizado  object 
 19  numAcordoQuadro           float64
 20  DescrAcordoQuadro       

Index(['idcontrato', 'nAnuncio', 'TipoAnuncio', 'idINCM', 'tipoContrato',
       'idprocedimento', 'tipoprocedimento', 'objectoContrato', 'descContrato',
       'adjudicante', 'adjudicatarios', 'dataPublicacao',
       'dataCelebracaoContrato', 'precoContratual', 'CPV', 'prazoExecucao',
       'LocalExecucao', 'fundamentacao', 'ProcedimentoCentralizado',
       'numAcordoQuadro', 'DescrAcordoQuadro', 'precoBaseProcedimento',
       'dataDecisaoAdjudicacao', 'dataFechoContrato', 'PrecoTotalEfetivo',
       'regime', 'justifNReducEscrContrato', 'tipoFimContrato',
       'CritMateriais', 'concorrentes', 'linkPecasProc', 'Observacoes',
       'ContratEcologico', 'fundamentAjusteDireto', 'Ano'],
      dtype='object')

In [42]:
(df['LocalExecucao'].str.lower() == "portugal").sum()

np.int64(129122)

In [43]:
percent_nulos = (df.isnull().sum() / len(df)) * 100

# Ordem decrescente
percent_nulos_ordenado = percent_nulos.sort_values(ascending=False)
print(percent_nulos_ordenado)

Observacoes                 94.975549
nAnuncio                    87.816819
TipoAnuncio                 87.816005
idINCM                      87.816005
numAcordoQuadro             87.740649
DescrAcordoQuadro           87.740649
justifNReducEscrContrato    82.739797
linkPecasProc               70.387675
tipoFimContrato             68.284049
dataFechoContrato           67.400390
fundamentAjusteDireto       66.790380
concorrentes                60.417655
dataDecisaoAdjudicacao       2.390782
fundamentacao                2.238497
dataCelebracaoContrato       2.237520
LocalExecucao                2.157770
adjudicatarios               0.111380
descContrato                 0.008084
regime                       0.002821
objectoContrato              0.002496
precoBaseProcedimento        0.001628
CPV                          0.001519
tipoContrato                 0.001465
adjudicante                  0.001085
tipoprocedimento             0.000271
idcontrato                   0.000000
idprocedimen

Apagar os registos em que as datas não fazem sentido


In [44]:
# Guardar ids com data de fecho anterior à celebração
ids_datas_invalidas = df.loc[df["dataFechoContrato"] < df["dataCelebracaoContrato"], "idcontrato"].tolist()
print(f"Contratos com datas inválidas: {len(ids_datas_invalidas)}")
# Remover do df
df = df[~df["idcontrato"].isin(ids_datas_invalidas)]
df = df[df["precoContratual"]>0]
print(f"Contratos restantes: {len(df)}")

Contratos com datas inválidas: 541
Contratos restantes: 1836902


In [45]:
colunas_remover = [
    "nAnuncio",
    "TipoAnuncio",
    "idINCM",
    "numAcordoQuadro",
    "dataFechoContrato",
    "tipoFimContrato",
    "precoBaseProcedimento",
    "PrecoTotalEfetivo",
]


In [46]:
df = df.drop(columns=colunas_remover, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1836902 entries, 0 to 1843245
Data columns (total 27 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   idcontrato                int64  
 1   tipoContrato              object 
 2   idprocedimento            int64  
 3   tipoprocedimento          object 
 4   objectoContrato           object 
 5   descContrato              object 
 6   adjudicante               object 
 7   adjudicatarios            object 
 8   dataPublicacao            object 
 9   dataCelebracaoContrato    object 
 10  precoContratual           float64
 11  CPV                       object 
 12  prazoExecucao             int64  
 13  LocalExecucao             object 
 14  fundamentacao             object 
 15  ProcedimentoCentralizado  object 
 16  DescrAcordoQuadro         object 
 17  dataDecisaoAdjudicacao    object 
 18  regime                    object 
 19  justifNReducEscrContrato  object 
 20  CritMateriais             obj

In [47]:
for col in df.select_dtypes("object"):
    if df[col].astype(str).str.contains("�").any():
        print("Problema encoding em:", col)
        
print(df.loc[df["adjudicatarios"].astype(str).str.contains("�", na=False), ["adjudicatarios"]].head(20))
print(df.loc[df["concorrentes"].astype(str).str.contains("�", na=False), ["concorrentes"]].head(20))

Problema encoding em: adjudicatarios
Problema encoding em: concorrentes
                                    adjudicatarios
1743020  - - Maria Beatriz Nunes Sampaio�b6&#x1F;R
                                    concorrentes
1743020  --Maria Beatriz Nunes Sampaio�b6&#x1F;R


In [48]:
for col in df.select_dtypes("object"):
    
    # remover caractere corrompido
    df[col] = df[col].str.replace("�", "", regex=False)
    
    # remover entidades HTML tipo &#x1F;
    df[col] = df[col].str.replace(r"&#x[0-9A-Fa-f]+;", "", regex=True)
    
    # remover caracteres invisíveis / não imprimíveis
    df[col] = df[col].str.replace(r"[^\x20-\x7EÀ-ÿ]", "", regex=True)

In [49]:
for col in df.select_dtypes(include="object"):
    if df[col].dropna().astype(str).str.startswith(" ").any() or df[col].dropna().astype(str).str.endswith(" ").any():
        print("Espaços extra em:", col)

Espaços extra em: objectoContrato
Espaços extra em: descContrato
Espaços extra em: adjudicante
Espaços extra em: adjudicatarios
Espaços extra em: CPV
Espaços extra em: fundamentacao
Espaços extra em: DescrAcordoQuadro
Espaços extra em: regime
Espaços extra em: justifNReducEscrContrato
Espaços extra em: concorrentes
Espaços extra em: linkPecasProc
Espaços extra em: Observacoes


ver categorias das variaveis


In [50]:
for col in df.select_dtypes(include="object"):
    print("\nColuna:", col)
    print(df[col].value_counts(dropna=False))


Coluna: tipoContrato
tipoContrato
Aquisição de bens móveis                                                            940582
Aquisição de serviços                                                               713157
Empreitadas de obras públicas                                                       143244
Locação de bens móveis                                                               19351
Aquisição de bens móveisAquisição de serviços                                        13129
                                                                                     ...  
Aquisição de serviçosEmpreitadas de obras públicasSociedade                              1
Aquisição de bens móveisConcessão de obras públicasSociedade                             1
Aquisição de bens móveisAquisição de serviçosEmpreitadas de obras públicasOutros         1
Aquisição de bens móveisLocação de bens móveisOutros                                     1
Aquisição de serviçosConcessão de obras públicasConcess

In [51]:
df["tipoContrato"].dropna().unique()

array(['Aquisição de bens móveis', 'Empreitadas de obras públicas',
       'Aquisição de serviços', 'Locação de bens móveis',
       'Aquisição de bens móveisAquisição de serviços', 'Outros',
       'Aquisição de serviçosLocação de bens móveis',
       'Concessão de serviços públicos',
       'Aquisição de bens móveisLocação de bens móveis',
       'Aquisição de bens móveisSociedade',
       'Aquisição de serviçosSociedade', 'Concessão de obras públicas',
       'Aquisição de serviçosEmpreitadas de obras públicas',
       'Aquisição de bens móveisConcessão de obras públicas',
       'Aquisição de bens móveisEmpreitadas de obras públicas',
       'Aquisição de bens móveisOutros',
       'Aquisição de bens móveisAquisição de serviçosEmpreitadas de obras públicas',
       'Concessão de obras públicasEmpreitadas de obras públicas',
       'Sociedade', 'Aquisição de serviçosOutros',
       'Aquisição de serviçosConcessão de obras públicas',
       'Aquisição de serviçosConcessão de serviços

In [52]:
df["objectoContrato"].dropna().unique()

array(['Prestação de serviços para Elaboração do Projecto dos Edifícios Destinados ao Centro de Incubação de Empresas',
       'Derivados do Plasma, AQ 2012/09',
       'Seringas e Contentores, AQ 2012/22 (Renov 2020014)', ...,
       'Aquisição de serviços especializados para a conceção e fornecimento de materiais personalizados (com logotipos de financiamento e nome/acrónimo do projeto) para construção do Kit pedagógico a ser distribuído pelos alunos e professores participantes nas atividades previstas na operação Conhecer para Preservar e Valorizar (acrónimo: BIONATURE_PNPG) - NORTE2030-FEDER-02711200.',
       'PROPOFOL (2%) 20 MG/ML EMUL INJ FR 50 ML IV',
       'Aquisição de serviços especializados para a conceção e criação de um documentário sobre o património natural do Parque Nacional da Peneda-Gerês (PNPG), que vise promover e valorizar a riqueza ecológica, paisagística e cultural desta área protegida, destacando a sua importância para a conservação da biodiversidade e o dese

In [53]:
set(df["objectoContrato"].dropna())

{'Serviço de acesso a plataforma eletrónica de gestão de catálogos e de fluxos documentais',
 'TACROLÍMUS 0.5 MG CÁPS LP',
 'EAD20250157 - Material Exclusivo robótica - Excelência robótica',
 'AQ- SERV PA- 25MI508',
 '2022/91 - DGEP - Construção da rampa na Rua Margarida Palla, em Algés',
 'aquisição de serviços de vigilância e segurança para o evento comemorativo dos 75 anos da WSI - WorldSkills ',
 'Aquisição de computadores e multifuncionais no âmbito do protocolo entre a RAM e a PSP (Processo 52/AD/2016)',
 'Fornecimento e suporte de plataformas de voz, monitorização, segurança e ligações cloud, dividido em 4 lotes:Lote 1: Fornecimento de Session border controllers e integração MS Teams com telefonia IP e equipamentos de vídeo e bolsa de horas de serviços de apoio;Lote 2: Expansão de licenciamento da plataforma PRTG Network Monitor, suporte e migração de outras plataformas de monitorização e bolsa de horas de serviços de apoio;Lote 3: Renovação de suporte para equipamentos de segur

Preços corrigidos para 2025

In [54]:
# garantir tipos corretos
df["precoContratual"] = pd.to_numeric(df["precoContratual"], errors="coerce")
df["dataPublicacao"] = pd.to_datetime(df["dataPublicacao"], errors="coerce")

# extrair ano do contrato
df["ano_contrato"] = df["dataPublicacao"].dt.year

# taxas anuais do IPC (Pordata, coluna Total)
taxas_ipc = {
    2015: 0.005,
    2016: 0.006,
    2017: 0.014,
    2018: 0.010,
    2019: 0.003,
    2020: 0.000,
    2021: 0.013,
    2022: 0.078,
    2023: 0.043,
    2024: 0.024,
    2025: 0.023,
}

def atualizar_para_2025(preco, ano):
    if pd.isna(preco) or pd.isna(ano):
        return np.nan
    
    ano = int(ano)
    
    # se o contrato for posterior a 2025, não ajusta
    if ano >= 2025:
        return preco
    
    fator = 1.0
    
    # multiplica do ano seguinte até 2025
    for a in range(ano + 1, 2026):
        fator *= (1 + taxas_ipc.get(a, 0))
    
    return preco * fator

df["precoContratual_2025"] = df.apply(
    lambda row: atualizar_para_2025(row["precoContratual"], row["ano_contrato"]),
    axis=1
)

print(df[["dataPublicacao", "ano_contrato", "precoContratual", "precoContratual_2025"]].head(10))

  dataPublicacao  ano_contrato  precoContratual  precoContratual_2025
0     2009-06-05          2009         12000.00          14869.407217
1     2015-01-02          2015            40.50             49.934576
2     2015-01-02          2015         21646.00          26688.489936
3     2015-01-02          2015         11647.50          14360.814309
4     2015-01-02          2015         16139.20          19898.867078
5     2015-01-02          2015         39400.00          48578.328720
6     2008-10-30          2008         36529.96          45264.904239
7     2009-05-07          2009          9760.00          12093.784537
8     2015-01-02          2015        600607.50         740520.521994
9     2015-01-02          2015        281238.60         346753.836535


Corrigir o prazoExecucao e precoContratual

In [55]:
df['setor_cpv'] = df['CPV'].astype(str).str.extract(r'^(\d{2})')
df['descricao_cpv'] = df['CPV'].astype(str).str.extract(r'^\d+(?:-\d+)?\s*-\s*(.*)$')
df['racio_preco_prazo'] = np.where(
    df['prazoExecucao'] > 0,
    df['precoContratual_2025'] / df['prazoExecucao'],
    np.nan
)

In [56]:
def calcular_limites(grupo):
    return pd.Series({
        "mediana": grupo["prazoExecucao"].median(),
        "limite_superior": grupo["prazoExecucao"].quantile(0.99),
        "descricao_cpv": grupo["descricao_cpv"].dropna().mode().iloc[0] if not grupo["descricao_cpv"].dropna().empty else None

    })

limites_cpv = df.groupby(df['setor_cpv']).apply(calcular_limites)

C:\Users\goncalo\AppData\Local\Temp\ipykernel_4120\86878064.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  limites_cpv = df.groupby(df['setor_cpv']).apply(calcular_limites)


In [57]:
df_prazo_igual_1 = df[df['prazoExecucao'] == 1]
df = df[~df["idcontrato"].isin(df_prazo_igual_1["idcontrato"])]
len(df)

1761141

In [58]:
def calcular_limites_precos(grupo):
    return pd.Series({
        "mediana_racio": grupo["racio_preco_prazo"].median(),
        "limite_superior_racio": grupo["racio_preco_prazo"].quantile(0.99),
        "descricao_cpv": grupo["descricao_cpv"].dropna().mode().iloc[0] if not grupo["descricao_cpv"].dropna().empty else None

    })
limites_cpv_precos = df.groupby(df['setor_cpv']).apply(calcular_limites_precos)


C:\Users\goncalo\AppData\Local\Temp\ipykernel_4120\3895538263.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  limites_cpv_precos = df.groupby(df['setor_cpv']).apply(calcular_limites_precos)


In [59]:
def contratos_acima_limite(df, limites_cpv):
    resultado = []

    for setor, grupo in df.groupby('setor_cpv'):
        if setor not in limites_cpv.index:
            continue

        limite = limites_cpv.loc[setor, 'limite_superior']

        print(f"Setor: {setor} | Limite: {limite}")

        ids_acima = grupo.loc[grupo['prazoExecucao'] > limite, 'idcontrato'].tolist()
        resultado.extend(ids_acima)

    total = len(resultado)
    perc = total / len(df) * 100
    print(f"Total de contratos acima do limite superior Prazos: {total}")
    print(f"Percentagem de contratos acima do limite superior Prazos: {perc:.3f}%")

    return resultado

ids_outliers = contratos_acima_limite(df, limites_cpv)
df = df[~df['idcontrato'].isin(ids_outliers)]


Setor: 03 | Limite: 1095.0
Setor: 09 | Limite: 1096.0
Setor: 14 | Limite: 1096.0
Setor: 15 | Limite: 1095.0
Setor: 16 | Limite: 1096.0
Setor: 18 | Limite: 1095.0
Setor: 19 | Limite: 1096.0
Setor: 22 | Limite: 1096.0
Setor: 24 | Limite: 1096.0
Setor: 30 | Limite: 1096.0
Setor: 31 | Limite: 1096.0
Setor: 32 | Limite: 1096.0
Setor: 33 | Limite: 1095.0
Setor: 34 | Limite: 1800.0
Setor: 35 | Limite: 1095.0
Setor: 37 | Limite: 847.9200000000037
Setor: 38 | Limite: 1095.0
Setor: 39 | Limite: 1095.0
Setor: 41 | Limite: 1556.0199999999918
Setor: 42 | Limite: 1096.0
Setor: 43 | Limite: 1096.0
Setor: 44 | Limite: 1096.0
Setor: 45 | Limite: 730.0
Setor: 48 | Limite: 1096.0
Setor: 50 | Limite: 1096.0
Setor: 51 | Limite: 1096.0
Setor: 55 | Limite: 1826.0
Setor: 60 | Limite: 1440.0
Setor: 63 | Limite: 1095.0
Setor: 64 | Limite: 1461.0
Setor: 65 | Limite: 1461.0
Setor: 66 | Limite: 1826.0
Setor: 70 | Limite: 5953.759999999984
Setor: 71 | Limite: 1096.0
Setor: 72 | Limite: 1096.0
Setor: 73 | Limite: 10

In [60]:
limites_cpv.sort_values('limite_superior', ascending=False)

,mediana,limite_superior,descricao_cpv
setor_cpv,,,
70,180.0,5953.76,Serviços de arrendamento ou locação de bens im...
55,49.0,1826.00,Serviços de fornecimento de refeições (catering)
66,365.0,1826.00,Serviços de seguros
34,60.0,1800.00,Equipamento e produtos auxiliares de transporte
41,45.0,1556.02,Água captada e tratada
65,365.0,1461.00,Serviços públicos
64,640.0,1461.00,Serviços de telecomunicações
75,349.0,1460.00,Serviços relacionados com a administração pública
60,180.0,1440.00,Serviços de transporte rodoviário de passageir...


In [61]:
def contratos_acima_limite_precos(df, limites_cpv_precos):
    resultado = []

    for setor, grupo in df.groupby(df['setor_cpv']):
        if setor not in limites_cpv_precos.index:
            continue
        limite = limites_cpv_precos.loc[setor, 'limite_superior_racio']
        ids_acima = grupo.loc[grupo['racio_preco_prazo'] > limite, 'idcontrato'].tolist()
        resultado.extend(ids_acima)

    total = len(resultado)
    perc = total / len(df) * 100
    print(f"Total de contratos acima do limite superior racio precos: {total}")
    print(f"Percentagem de contratos acima do limite superior racio precos: {perc:.3f}%")

    return resultado

ids_outliers_precos = contratos_acima_limite_precos(df, limites_cpv_precos)
df = df[~df['idcontrato'].isin(ids_outliers_precos)]

Total de contratos acima do limite superior racio precos: 16988
Percentagem de contratos acima do limite superior racio precos: 0.972%


In [62]:
limites_cpv_precos.sort_values('limite_superior_racio', ascending=False)

,mediana_racio,limite_superior_racio,descricao_cpv
setor_cpv,,,
92,188.247518,23722.471465,"Serviços recreativos, culturais e desportivos"
55,282.151371,19032.000164,"Serviços de hotelaria, restauração e comércio ..."
09,172.406797,18241.679205,Electricidade
34,338.585353,13591.191023,Equipamento e produtos auxiliares de transporte
35,231.816667,13280.449907,Equipamento de segurança e protecção
32,520.565255,12974.325278,"Equipamento de rádio, televisão, comunicação, ..."
16,507.188014,12788.250390,Maquinaria agrícola
79,100.036612,12277.769488,"Serviços a empresas: direito, comercialização,..."
43,545.027342,12178.622317,Equipamento para parques e áreas de recreação


In [63]:
df = pd.concat([df, df_prazo_igual_1]).reset_index(drop=True)
len(df)


1806404

In [64]:
ids_repetidos = df[df["idcontrato"].duplicated(keep=False)]
df = df[~df["idcontrato"].isin(ids_repetidos["idcontrato"])]


In [65]:
ids_repetidos.groupby('idcontrato')['dataCelebracaoContrato'].nunique().gt(1).sum()

ids_repetidos = (
    ids_repetidos
    .sort_values('dataCelebracaoContrato')
    .drop_duplicates('idcontrato', keep='last')
)
print(len(ids_repetidos))

2911


In [66]:
antes = len(df)
df = pd.concat([df, ids_repetidos]).reset_index(drop=True)
print(f"Contratos adicionados: {len(df) - antes}")
print(f"Contratos totais: {len(df)}")

Contratos adicionados: 2911
Contratos totais: 1803493


Limpeza dos campos de adjudicatários, adjudicantes e concorrentes

In [67]:
def separar_entidades_por_nif(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto).strip()
    if texto == '':
        return pd.NA

    # converter sequências escapadas em quebras reais
    texto = texto.replace('\\n', '\n').replace('\\r', '\n').replace('\\t', ' ')

    # normalizar espaços sem destruir \n
    texto = re.sub(r'[ \t]+', ' ', texto)

    # quebrar antes de cada novo NIF, exceto no início
    texto = re.sub(r'(?<!^)(\d{9})', r'\n\1', texto)

    partes = [p.strip() for p in texto.split('\n') if p.strip()]

    return '\n'.join(partes) if partes else pd.NA

df['concorrentes'] = df['concorrentes'].apply(separar_entidades_por_nif)
df['adjudicatarios'] = df['adjudicatarios'].apply(separar_entidades_por_nif)
df['adjudicante'] = df['adjudicante'].apply(separar_entidades_por_nif)


In [68]:
def limpar_texto_base(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto).strip()
    if texto == '':
        return pd.NA

    # maiúsculas
    texto = texto.upper()

    # remover acentos
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

    # corrigir html
    texto = texto.replace('&AMP;', '&')

    # normalizações específicas
    texto = texto.replace('B BRAUN', 'BBRAUN')
    texto = re.sub(r'\bLAB\b', 'LABORATORIOS', texto)

    # remover pontuação
    texto = re.sub(r'[.,;:\-_/\\()]+', ' ', texto)

    texto = re.sub(r'\b(LDA|LDA\.|SA|S A|S\.A\.|UNIPESSOAL|S A U|SAU|SL|S L|SUCURSAL)\b', '', texto)

    # normalizar espaços
    texto = re.sub(r'[ \t]+', ' ', texto).strip()

    return texto if texto != '' else pd.NA


Funções para limpar nomes com NIF

In [69]:

def limpar_nome_com_nif(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()
    if valor == '':
        return pd.NA

    valor = valor.replace('\\n', '\n').replace('\\r', '\n').replace('\\t', ' ')
    valor = re.sub(r'\s+', ' ', valor).strip()

    match = re.match(r'^(\d{9})\s*[,;:\-]?\s*(.*)$', valor)
    if match:
        nif = match.group(1)
        nome = match.group(2).strip()
        nome = limpar_texto_base(nome)
        if pd.isna(nome) or nome == '':
            return nif
        return f'{nif}, {nome}'

    nome = limpar_texto_base(valor)
    return nome


def limpar_lista_nomes_com_nif(valor):
    """
    Para colunas tipo concorrentes, preservando \n.
    Divide por linhas e também por NIFs repetidos na mesma linha.
    """
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()
    if valor == '':
        return pd.NA

    itens_limpos = []

    linhas = valor.split('\n')

    for linha in linhas:
        linha = linha.strip()
        if not linha:
            continue

        # separa antes de cada NIF de 9 dígitos
        partes = re.split(r'(?=\b\d{9}\s)', linha)

        for parte in partes:
            parte = parte.strip()
            if not parte:
                continue

            limpo = limpar_nome_com_nif(parte)
            if pd.notna(limpo) and limpo != '':
                itens_limpos.append(limpo)

    # remover duplicados preservando ordem
    itens_limpos = list(dict.fromkeys(itens_limpos))

    return '\n'.join(itens_limpos) if itens_limpos else pd.NA


def criar_mapa_primeiro_nome_por_nif(serie):
    tmp = serie.str.extract(r'^(\d{9})[\s,]+(.*)$')
    tmp.columns = ['nif', 'nome']

    mapa = (
        tmp.dropna(subset=['nif', 'nome'])
        .drop_duplicates(subset='nif', keep='first')
        .set_index('nif')['nome']
    )

    return mapa


def aplicar_mapa_nif_nome(valor, mapa_nomes):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()
    match = re.match(r'^(\d{9})[\s,]+(.*)$', valor)
    if not match:
        return valor

    nif = match.group(1)

    if nif in mapa_nomes.index and pd.notna(mapa_nomes[nif]):
        return f'{nif}, {mapa_nomes[nif]}'

    return valor

def aplicar_mapa_nif_nome_em_lista(valor, mapa_nomes):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).replace('\\n', '\n').replace('\\r', '\n').replace('\\t', ' ')
    linhas = valor.split('\n')
    novas_linhas = []

    for linha in linhas:
        linha = linha.strip()
        if not linha:
            continue

        nova = aplicar_mapa_nif_nome(linha, mapa_nomes)
        if pd.notna(nova) and nova != '':
            novas_linhas.append(nova)

    novas_linhas = list(dict.fromkeys(novas_linhas))

    return '\n'.join(novas_linhas) if novas_linhas else pd.NA

limpar as colunas

In [70]:
df['adjudicatarios'] = df['adjudicatarios'].apply(limpar_nome_com_nif)


df['concorrentes'] = df['concorrentes'].apply(limpar_lista_nomes_com_nif)

df['adjudicante'] = df['adjudicante'].apply(limpar_nome_com_nif)

Normalização dos adjudicatários por NIF

In [71]:
mapa_nomes_adjudicatarios = criar_mapa_primeiro_nome_por_nif(df['adjudicatarios'])

df['adjudicatarios'] = df['adjudicatarios'].apply(
    lambda x: aplicar_mapa_nif_nome(x, mapa_nomes_adjudicatarios)
)


Normalização dos concorrentes

In [72]:

df['concorrentes'] = df['concorrentes'].apply(
    lambda x: aplicar_mapa_nif_nome_em_lista(x, mapa_nomes_adjudicatarios)
)


Normalização dos adjudicantes

In [73]:

mapa_nomes_adjudicante = criar_mapa_primeiro_nome_por_nif(df['adjudicante'])

df['adjudicante'] = df['adjudicante'].apply(
    lambda x: aplicar_mapa_nif_nome(x, mapa_nomes_adjudicante)
)


In [74]:
def remover_adjudicatario_dos_concorrentes(row):

    if pd.isna(row['adjudicatarios']) or pd.isna(row['concorrentes']):
        return row['concorrentes']

    vencedor = row['adjudicatarios'].strip()

    concorrentes = [
        c.strip()
        for c in str(row['concorrentes']).split('\n')
        if c.strip() != ''
    ]

    # remover adjudicatário dos concorrentes
    concorrentes_limpos = [
        c for c in concorrentes
        if c != vencedor
    ]

    # Se não sobrar nada → NaN
    if len(concorrentes_limpos) == 0:
        return pd.NA

    # Voltar a juntar com newline
    return '\n'.join(concorrentes_limpos)


# Aplicar ao dataframe
df['concorrentes'] = df.apply(
    remover_adjudicatario_dos_concorrentes,
    axis=1
)

In [75]:

df["dataCelebracaoContrato"] = pd.to_datetime(df["dataCelebracaoContrato"], errors="coerce")

datas_al = pd.to_datetime([
    "2017-10-01",
    "2021-09-26",
    "2025-10-12"
])

datas_ar = pd.to_datetime([
    "2015-10-04",
    "2019-10-06",
    "2022-01-30",
    "2024-03-10",
    "2025-05-18"
])

datas_todas = datas_al.tolist() + datas_ar.tolist()



df["janela_eleicoes"] = "não"
df["AR"] = "não"
df["AL"] = "não"


for data_eleicao in datas_todas:
    mask_antes = (
        (df["dataCelebracaoContrato"] >= data_eleicao - pd.DateOffset(months=3)) &
        (df["dataCelebracaoContrato"] < data_eleicao)
    )

    mask_depois = (
        (df["dataCelebracaoContrato"] >= data_eleicao) &
        (df["dataCelebracaoContrato"] <= data_eleicao + pd.DateOffset(months=3))
    )

    df.loc[mask_antes, "janela_eleicoes"] = "3 meses antes"
    df.loc[mask_depois, "janela_eleicoes"] = "3 meses depois"


for data_eleicao in datas_ar:
    mask_ar = (
        (df["dataCelebracaoContrato"] >= data_eleicao - pd.DateOffset(months=3)) &
        (df["dataCelebracaoContrato"] <= data_eleicao + pd.DateOffset(months=3))
    )

    df.loc[mask_ar, "AR"] = "sim"



for data_eleicao in datas_al:
    mask_al = (
        (df["dataCelebracaoContrato"] >= data_eleicao - pd.DateOffset(months=3)) &
        (df["dataCelebracaoContrato"] <= data_eleicao + pd.DateOffset(months=3))
    )

    df.loc[mask_al, "AL"] = "sim"



print(df["janela_eleicoes"].value_counts(dropna=False))
print(df["AR"].value_counts(dropna=False))
print(df["AL"].value_counts(dropna=False))


janela_eleicoes
não               1156969
3 meses antes      330008
3 meses depois     316516
Name: count, dtype: int64
AR
não    1352971
sim     450522
Name: count, dtype: int64
AL
não    1560633
sim     242860
Name: count, dtype: int64


In [76]:
df.to_csv("contratos_limpo.csv", index=False)